In [1]:
import ast
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

**Load and inspect the dataset**

In [2]:
file_path = 'dblp-v10.csv'
raw_df = pd.read_csv(file_path, low_memory=False)
raw_df.head()

,abstract,authors,n_citation,references,title,venue,year,id
0,"In this paper, a robust 3D triangular mesh wat...","['S. Ben Jabra', 'Ezzeddine Zagrouba']",50,"['09cb2d7d-47d1-4a85-bfe5-faa8221e644b', '10aa...",A new approach of 3D watermarking based on ima...,international symposium on computers and commu...,2008,4ab3735c-80f1-472d-b953-fa0557fed28b
1,We studied an autoassociative neural network w...,"['Joaquín J. Torres', 'Jesús M. Cortés', 'Joaq...",50,"['4017c9d2-9845-4ad2-ad5b-ba65523727c5', 'b118...",Attractor neural networks with activity-depend...,Neurocomputing,2007,4ab39729-af77-46f7-a662-16984fb9c1db
2,It is well-known that Sturmian sequences are t...,"['Genevi eve Paquin', 'Laurent Vuillon']",50,"['1c655ee2-067d-4bc4-b8cc-bc779e9a7f10', '2e4e...",A characterization of balanced episturmian seq...,Electronic Journal of Combinatorics,2007,4ab3a4cf-1d96-4ce5-ab6f-b3e19fc260de
3,One of the fundamental challenges of recognizi...,"['Yaser Sheikh', 'Mumtaz Sheikh', 'Mubarak Shah']",221,"['056116c1-9e7a-4f9b-a918-44eb199e67d6', '05ac...",Exploring the space of a human action,international conference on computer vision,2005,4ab3a98c-3620-47ec-b578-884ecf4a6206
4,This paper generalizes previous optimal upper ...,"['Efraim Laksman', 'Håkan Lennerstad', 'Magnus...",0,"['01a765b8-0cb3-495c-996f-29c36756b435', '5dbc...",Generalized upper bounds on the minimum distan...,Ima Journal of Mathematical Control and Inform...,2015,4ab3b585-82b4-4207-91dd-b6bce7e27c4e


In [3]:
df = raw_df.copy()
df = df.drop_duplicates()

text_columns = ['abstract', 'authors', 'references', 'title', 'venue']
for column in text_columns:
    if column in df.columns:
        df[column] = df[column].fillna('')

for column in ['n_citation', 'year']:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')
        df[column] = df[column].fillna(df[column].median())

df = df.replace([np.inf, -np.inf], np.nan).fillna('')
df = df.reset_index(drop=True)
df.shape

(1000000, 8)

**Data inspection after cleaning**

This comes after cleaning so you can verify the cleaned dataframe before feature extraction.

In [4]:
df
df.head()
df.tail()
df.info()
df.isnull().sum().sum()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column      Non-Null Count    Dtype
---  ------      --------------    -----
 0   abstract    1000000 non-null  str  
 1   authors     1000000 non-null  str  
 2   n_citation  1000000 non-null  int64
 3   references  1000000 non-null  str  
 4   title       1000000 non-null  str  
 5   venue       1000000 non-null  str  
 6   year        1000000 non-null  int64
 7   id          1000000 non-null  str  
dtypes: int64(2), str(6)
memory usage: 1.3 GB


0

In [5]:
import re

def author_count(value):
    if isinstance(value, str) and value.strip():
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, (list, tuple, set, dict)):
                return len(parsed)
        except (ValueError, SyntaxError):
            return len([part for part in value.split(',') if part.strip()])
    return 0

stop_words = {
    'the', 'and', 'for', 'with', 'from', 'this', 'that', 'are', 'was', 'were',
    'into', 'using', 'based', 'paper', 'study', 'analysis', 'method', 'model',
    'data', 'approach', 'system', 'results', 'research', 'can', 'have', 'has',
    'theory', 'between', 'over', 'under', 'their', 'there', 'also', 'more', 'than'
}

def keyword_count(value):
    text = str(value).lower()
    tokens = re.findall(r'[a-z]{5,}', text)
    keywords = {token for token in tokens if token not in stop_words}
    return len(keywords)

df['title_length'] = df['title'].astype(str).str.len()
df['abstract_length'] = df['abstract'].astype(str).str.len()
df['author_count'] = df['authors'].apply(author_count)
df['keyword_count'] = (df['title'].astype(str) + ' ' + df['abstract'].astype(str)).apply(keyword_count)
df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(df['year'].median())
df['n_citation'] = pd.to_numeric(df['n_citation'], errors='coerce').fillna(df['n_citation'].median())
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
df[['title_length', 'abstract_length', 'author_count', 'keyword_count', 'year', 'n_citation']].head()

,title_length,abstract_length,author_count,keyword_count,year,n_citation
0,61,511,2,33,2008,50
1,93,1021,4,69,2007,50
2,52,711,2,30,2007,50
3,37,1019,3,64,2005,221
4,67,201,3,19,2015,0


In [6]:
model_df = df[df['abstract_length'] > 0].copy()
model_df = model_df.sample(n=min(100000, len(model_df)), random_state=42).copy()
model_df = model_df.sort_values('year').reset_index(drop=True)
model_df[['abstract_length', 'author_count', 'keyword_count', 'year', 'n_citation']].head()

,abstract_length,author_count,keyword_count,year,n_citation
0,1139,1,67,1953,10
1,1927,1,95,1953,50
2,416,1,24,1955,0
3,1270,1,60,1955,88
4,405,2,30,1955,50


Simple Linear Regression


In [7]:
X_linear = model_df[['abstract_length']]
y = model_df['n_citation']

X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_linear, y, test_size=0.2, random_state=42
)

linear_model = LinearRegression()
linear_model.fit(X_train_lin, y_train_lin)

y_pred_lin = linear_model.predict(X_test_lin)
lin_mae = mean_absolute_error(y_test_lin, y_pred_lin)
lin_rmse = np.sqrt(mean_squared_error(y_test_lin, y_pred_lin))
lin_r2 = r2_score(y_test_lin, y_pred_lin)

linear_model.coef_, linear_model.intercept_, lin_mae, lin_rmse, lin_r2

(array([0.00620262]),
 33.979095923724934,
 42.619495606633336,
 201.92799225756067,
 -0.000151658388353626)

**Predict citation count for a sample year**

In [8]:
linear_model.predict([[1200]])

c:\Users\UTHANDAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([41.42223463])

Multivariate Regression


In [9]:
feature_columns = ['abstract_length', 'author_count', 'year', 'keyword_count']
X_multi = model_df[feature_columns]
y_multi = model_df['n_citation']

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

multi_model = LinearRegression()
multi_model.fit(X_train_multi, y_train_multi)

y_pred_multi = multi_model.predict(X_test_multi)
multi_mae = mean_absolute_error(y_test_multi, y_pred_multi)
multi_rmse = np.sqrt(mean_squared_error(y_test_multi, y_pred_multi))
multi_r2 = r2_score(y_test_multi, y_pred_multi)

multi_model.coef_, multi_model.intercept_, multi_mae, multi_rmse, multi_r2

(array([-3.18209478e-03,  2.31009030e+00, -4.26944136e+00,  4.48298772e-01]),
 8587.980200124704,
 39.639630497125836,
 199.59041370468265,
 0.022870417309969304)

In [10]:
sample_record = pd.DataFrame([{
    'abstract_length': 800,
    'author_count': 3,
    'year': 2010,
    'keyword_count': 25
}])
multi_model.predict(sample_record)

array([21.99512129])